# AIMDL API Endpoints

This notebook calls each `aimdl/` REST endpoint directly using the Girder client, showing what parameters each accepts and what the raw response looks like.

| Endpoint | Description |
|---|---|
| `GET /aimdl/count` | Count data files per type across the collection |
| `GET /aimdl/datatype` | List all available data types |
| `GET /aimdl/datafiles` | Query items by data type (pagination, sort, filter, extra fields) |
| `GET /aimdl/partition` | List partitions that have data for a given type |
| `GET /aimdl/partition/details` | Get Dagster partition details for a specific IGSN |

## Setup

Authenticate directly with the Girder client. Reads `GIRDER_API_URL` and `GIRDER_API_KEY` from the `.env` file at the project root.

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from girder_client import GirderClient

load_dotenv(find_dotenv())

gc = GirderClient(apiUrl=os.environ['GIRDER_API_URL'])
gc.authenticate(apiKey=os.environ['GIRDER_API_KEY'])
login = gc.get('user/me')['login']
print('Authenticated as:', login)

Authenticated as: arachid1


## `GET /aimdl/count`

Returns the number of data files per type in the AIMDL collection. Accepts optional `baseParentId` and `baseParentType` to scope the count to a specific Girder collection.

In [2]:
counts = gc.get('aimdl/count')
print(json.dumps(counts, indent=2))

{
  "pdv_alpss_output": 26376,
  "pdv_alpss_result": 3297,
  "pdv_experiment_log": 360,
  "pdv_trace": 3691,
  "unclassified": 45349,
  "xrd_calibrant_derived": 48,
  "xrd_calibrant_raw": 32,
  "xrd_derived": 52593,
  "xrd_metadata": 1256,
  "xrd_raw": 32111,
  "xrf_raw": 14947
}


## `GET /aimdl/datatype`

Returns an array of all data type strings registered in the AIMDL collection. No parameters required.

In [3]:
data_types = gc.get('aimdl/datatype')
print(json.dumps(data_types, indent=2))

[
  null,
  "pdv_alpss_output",
  "pdv_alpss_result",
  "pdv_experiment_log",
  "pdv_trace",
  "xrd_calibrant_derived",
  "xrd_calibrant_raw",
  "xrd_derived",
  "xrd_metadata",
  "xrd_raw",
  "xrf_raw"
]


## `GET /aimdl/datafiles`

The main query endpoint. `dataType` is required; everything else is optional.

| Parameter | Default | Description |
|---|---|---|
| `dataType` | — | **Required.** The data type to filter by |
| `limit` | 50 | Page size |
| `offset` | 0 | Pagination offset |
| `sort` | `lowerName` | Field to sort by |
| `sortdir` | 1 | 1 = ascending, -1 = descending |
| `extraFields` | — | JSON array of dotted field paths to include (e.g. `["meta.prov", "meta.runId"]`) |
| `filters` | — | JSON object of additional server-side filters |
| `baseParentId` / `baseParentType` | — | Scope to a specific Girder collection |

In [4]:
# Basic call — fetch 3 pdv_alpss_result items
items = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 3,
})

print(f'Fetched {len(items)} items')
print()
print('First item (default fields):')
print(json.dumps(items[0], indent=2))

Fetched 3 items

First item (default fields):
{
  "_id": "6a296cb2ac4f2d98ffa04626",
  "_modelType": "item",
  "baseParentId": "665de536bcc722774ce53754",
  "baseParentType": "collection",
  "created": "2026-06-10T13:54:58.824000+00:00",
  "creatorId": "6a280b9d20783bdc81a10fc2",
  "folderId": "6a296cb2ac4f2d98ffa0460f",
  "meta": {
    "data_type": "pdv_alpss_result",
    "igsn": "JHAMAA00001-S5R4C3"
  },
  "name": "C1--20250807--00001-results.csv",
  "size": 1418
}


### Sorting

Use `sort` and `sortdir` to control result order. `sortdir=-1` gives descending order (most recently created first).

In [5]:
recent = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 5,
    'sort': 'created',
    'sortdir': -1,
})

print('5 most recently created pdv_alpss_result items:')
for item in recent:
    print(' ', item['name'], '—', item.get('created', ''))

5 most recently created pdv_alpss_result items:
  JHAMAC00003-S1R4C2_2026-06-12_00-40-09_shot15-results.csv — 2026-06-11T20:43:07.002000+00:00
  JHAMAC00003-S1R4C2_2026-06-12_00-40-01_shot14-results.csv — 2026-06-11T20:43:01.154000+00:00
  JHAMAC00003-S1R4C2_2026-06-12_00-39-49_shot13-results.csv — 2026-06-11T20:42:55.591000+00:00
  JHAMAC00003-S1R4C2_2026-06-12_00-39-21_shot12-results.csv — 2026-06-11T20:40:47.280000+00:00
  JHAMAC00003-S1R4C2_2026-06-12_00-39-14_shot11-results.csv — 2026-06-11T20:40:41.428000+00:00


### Extra fields

`extraFields` is a JSON-encoded array of dotted field paths. Use it to include metadata not returned by default, such as provenance info or run IDs.

In [6]:
items_with_prov = gc.get('aimdl/datafiles', parameters={
    'dataType': 'pdv_alpss_result',
    'limit': 1,
    'extraFields': '["meta.prov", "meta.runId"]',
})

item = items_with_prov[0]
print('meta with extra fields:')
print(json.dumps(item.get('meta', {}), indent=2))

meta with extra fields:
{
  "data_type": "pdv_alpss_result",
  "igsn": "JHAMAA00001-S5R4C3",
  "prov": {
    "wasDerivedFrom": "6894efd2d48b09febbb57e48",
    "wasGeneratedBy": "alpss_dagster/0.1.1 alpss/1.7.1"
  },
  "runId": "ac25c1d9-ebdf-4bdc-9ce1-b692c2e78a7e"
}


## `GET /aimdl/partition`

Returns a dict of partition keys (formatted as `IGSN//Date`) mapped to checksums.

In [21]:
partitions = gc.get('aimdl/partition', parameters={
    'dataType': 'pdv_alpss_result',
})

# The response is a dict keyed by IGSN — use list() to get the keys as a list
print('Response type:', type(partitions).__name__)
igsn_list = list(partitions.keys())
print(f'Found {len(igsn_list)} partitions with pdv_alpss_result data')
print('First 5:', igsn_list[:5])

Response type: dict
Found 175 partitions with pdv_alpss_result data
First 5: ['JHACRD00005-S10//2026-03-20', 'JHACRD00007-S05//2026-03-20', 'JHAMAA00001-S1R4C1//2025-08-11', 'JHAMAA00001-S1R5C1//2025-08-11', 'JHAMAA00001-S2R3C3//2025-12-12']


The optional `since` parameter (ISO 8601 date string) limits results to partitions updated after that date — handy for incremental syncs

In [8]:
# The `since` parameter filters to items updated after a date (ISO 8601)
recent_partitions = gc.get('aimdl/partition', parameters={
    'dataType': 'pdv_alpss_result',
    'since': '2025-01-01T00:00:00',
})

print(f'IGSNs with pdv_alpss_result updated since 2025-01-01: {len(recent_partitions)}')

IGSNs with pdv_alpss_result updated since 2025-01-01: 175


### Understanding Partition Keys

Each partition key combines an IGSN (sample identifier) and experiment date, formatted as `IGSN//Date`. For example: `JHACRD00005-S10//2026-03-20`. The value is a checksum of that partition's content, enabling you to:
- Track which samples have data for a given data type
- Detect when data changes (checksums will differ)
- Fetch data incrementally organized by sample and date

In [24]:
# Inspect partition structure
print('Partition key format: IGSN//Date → Checksum')
for key, checksum in list(partitions.items())[:3]:
    igsn, date = key.split('//')
    print(f'  {igsn} ({date}) → {checksum}')

Partition key format: IGSN//Date → Checksum
  JHACRD00005-S10 (2026-03-20) → e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  JHACRD00007-S05 (2026-03-20) → e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
  JHAMAA00001-S1R4C1 (2025-08-11) → e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855


## `GET /aimdl/partition/details`

Returns Dagster partition details for a specific IGSN. The `key` parameter is an IGSN string from the `/aimdl/partition` response. Use this to inspect what data has been processed for a single sample.

In [9]:
# Use the first IGSN from the list above
first_igsn = igsn_list[0]
print('Fetching details for IGSN:', first_igsn)

details = gc.get('aimdl/partition/details', parameters={
    'key': first_igsn,
    'dataType': 'pdv_alpss_result',
})

print(json.dumps(details, indent=2))

Fetching details for IGSN: JHACRD00005-S10//2026-03-20
[
  {
    "_id": "6a296b3e0b116ccbfdcc1937",
    "_modelType": "item",
    "baseParentId": "665de536bcc722774ce53754",
    "baseParentType": "collection",
    "created": "2026-06-10T13:48:46.026000+00:00",
    "creatorId": "6a280b9d20783bdc81a10fc2",
    "description": "",
    "folderId": "6a296b3d0b116ccbfdcc1920",
    "meta": {
      "alpss_output_name": "results",
      "data_type": "pdv_alpss_result",
      "experiment_date": "2026-03-20",
      "igsn": "JHACRD00005-S10",
      "prov": {
        "wasDerivedFrom": "69bdbbcc7590bb7e1ed25708",
        "wasGeneratedBy": "alpss_dagster/0.1.1 alpss/1.7.1"
      },
      "runId": "cb844201-eabd-41f4-b614-0e8293d8b7cc"
    },
    "name": "JHACRD00005-S10_2026-03-20_22-12-07_shot01_ch1-results.csv",
    "size": 1349,
    "updated": "2026-06-11T16:40:08.907000+00:00"
  },
  {
    "_id": "6a296b4540fe08cc73d0fd7a",
    "_modelType": "item",
    "baseParentId": "665de536bcc722774ce53754",
